# Load deps

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import requests, time, json
from pprint import pprint
from minsearch import Index
from sqlitesearch import TextSearchIndex
from src.ingest import load_faq_data, build_index, build_sqlite_index
from src.rag_helper import RAGBase

load_dotenv()

# Create OpenAI client

Check that the OpenAI client works

In [ ]:
openai_client = OpenAI()

# Plain LLMs lack our data

First, let's define a function to talk to the LLM:

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

This is our black box - text goes in, text comes out.

Let's test it:

In [ ]:
llm("Hey, what's up?")

Ask it a course-specific question:

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. But our courses are not in the training data.

# Adding context manually

More context can fix this. The FAQ website has questions and answers about our courses.

Copy some of that content into the prompt:

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""


prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

# Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.

In code, it looks like this:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```
That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:
-   search
-   prompt
-   LLM


The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, so search quality matters a lot for RAG.

The database and the LLM can be anything. Because each piece is independent, RAG stays flexible.
- for now, we use minsearch and then sqlitesearch for search, and OpenAI for the LLM
- To use Anthropic instead of OpenAI, you swap the LLM call.
- To use Elasticsearch instead of minsearch, you swap the search call. Nothing else changes.

## The Course FAQ Dataset

Before we build the RAG pipeline, let's get familiar with the data we'll use as our knowledge base.

The FAQ data is available as JSON from the DataTalks.Club website.

Let's fetch it:

In [ ]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
pprint(courses_raw)

Let's fetch all the FAQ documents from all courses:

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

In [ ]:
print(len(documents))
pprint(documents[3])

## Using FAQ data

In the RAG pipeline, this dataset is our knowledge base:

1.  We index all the documents (the search step)
2.  When a student asks a question, we search the index
3.  The search returns the most relevant FAQ entries
4.  We give those entries to the LLM as context
5.  The LLM generates an answer based on the context

- The `question` and `answer` fields contain the text we'll search through.
- The `course` field lets us filter by course. For example, if a student asks about the data engineering course, we skip results from the ML course.
- The `section` field helps with ranking - knowing which part of the course a question belongs to is useful context.

### A note on data preparation

- In our case, the data is already prepared in a convenient JSON format. So we don't need to do much to get it ready.
- Don't let that fool you. In reality, data preparation is often the most time-consuming part of building a RAG system. You may need to scrape websites, parse PDFs, and clean and chunk documents. That work isn't visible here.

## Search Engine

At its core, every search engine does the same thing. It takes a query, scores every document for similarity, and returns the top results.

Formally, there is a similarity function:

```python
score = sim(query, document)
```

For each document in the database, you compute this score. Then you rank all documents by score and return the top N. What makes a search engine different from another search engine is what `sim` actually computes.

-   text/lexical search: `sim` counts how many words the query and the document share. It looks at the surface form, the actual words, and matches them exactly.
-   vector/semantic search: `sim` compares the meaning of the query and the document. Same function, different similarity measure.

Consider these two questions:
-   "Can I still join the course after the start date?"
-   "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not "enroll", "course" is absent, "start date" is not "late". A text search engine would struggle to match them, because it only sees words.

**Searching matters** because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow. The model would get confused with too much data. Search finds the most relevant documents to send instead.

### Indexing with minsearch

There are many search libraries you can use:
- Apache Lucene,
- Elasticsearch,
- Solr,
- and others.
But these are somewhat heavy. For example, to run Elasticsearch, you need to start a Docker container.

[minsearch](https://github.com/alexeygrigorev/minsearch) is a simple in-memory search engine. It's lightweight, so it runs anywhere Python runs, including Google Colab where you can't start a Docker container. It's a toy implementation, not production ready, but it illustrates how search engines work and it gives good results.

[Alexey](https://github.com/alexeygrigorev) wrote this library and maintains it. It started as a single Python file in the first edition of LLM Zoomcamp. He wrote it as part of the [Build a Search Engine](https://www.youtube.com/watch?v=nMrGK5QgPVE) workshop (see the [code](https://github.com/alexeygrigorev/build-your-own-search-engine)).

- The concepts in minsearch are the same as in Elasticsearch (which comes from Lucene): text fields, keyword fields, boosting, filtering. So what you learn here transfers directly.
- We'll index the `question`, `section`, and `answer` fields as text (they'll be tokenized and ranked), and the `course` field as a keyword (for filtering).
- The index tokenizes text fields and treats keyword fields as exact strings.
- Text fields are the fields you search through. When you type a query, the search engine looks at these fields and tokenizes them. It splits text into words, lowercases them, removes stop words, and ranks the results by relevance. So `question`, `section`, and `answer` are text fields.
- Keyword fields are for exact matching. Think of a SQL query like `SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'`. The results must come from the specified course, no matter what ranking or boosting you do for text fields. You use keyword fields to restrict the search space to a particular subset. In our case, we have four courses. Say you're taking the LLM course and ask a question. You don't want answers from the MLOps or machine learning courses mixed in.

In [ ]:
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

### Trying a search

Let's try a search with the question we used before:
- We use `boost_dict` to give the `question` field more weight (2.0) and `section` less weight (0.5).
    - All fields have a default boost of 1.0
    - Query words appearing in the question field is a stronger signal than them appearing in the section name.
- We use `filter_dict` to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
pprint([doc["question"] for doc in search_results])

In [ ]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
pprint([doc["question"] for doc in results])

### Our first search function

Let's wrap the search in a `search` function - the first component of our RAG pipeline:

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
search_results = search(question)
pprint([doc["question"] for doc in search_results])

## Building the Prompt

The LLM doesn't see our documents unless we pass them in. So we need to build a prompt that includes the user's question and the search
results.

When we build AI systems, we usually split the prompt into two parts:
- Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
- User prompt: this changes with every request. It carries the actual question and the retrieved context.

### Instructions

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate answers. If the answer is not found in the context,
respond with "I don't know."
"""

### The user prompt template

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

### Building the context

- The `context` is a formatted string with all the search results.
- Each document becomes a block with the section, question, and answer. This format makes it easy for the LLM to read. We turned a list of dictionaries into one string. It's a small preprocessing step before we send the data to the LLM.

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

### Building the prompt

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

Let's try it:

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

## The LLM

The last component of our RAG pipeline is the LLM. It takes the prompt we built and generates an answer.

### Sending the prompt to the LLM

We have the prompt from the previous section. We send it to the LLM:

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

### Exploring the response

The response is a Pydantic object. The answer is in `response.output` - a list of output items.
The first one is the message:

In [ ]:
print(response.output[0].content[0].text)
print("========================")
print(response.output_text)

The usage counts tell you how many tokens the request consumed:

In [ ]:
print(response.usage)

### Calculating the price

We're using [gpt-5.4-mini](https://developers.openai.com/api/docs/models/gpt-5.4-mini):

-   Input: $0.75 per million tokens
-   Output: $4.50 per million tokens

Let's calculate the cost of the request we just made:

In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

print(cost)

### Message history

Previously we sent only one string as input and got back a response. In practice, you typically send a message history - a list of messages where each message has a role.

Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.

We won't build a multi-turn chat here. But we still use this message format to separate our instructions from the user prompt.

We send two messages:

- `developer` - system-level instructions (how the LLM should behave)
    - OpenAI accepts both `developer` and `system` for the instruction role.
- `user` - the actual prompt with the question and context

In [ ]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [ ]:
print(response.output_text)
print("========================")
print(response.usage)

### The LLM function

We can now put this together into an updated `llm` function. It now takes both instructions and the user prompt:

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

## Full RAG

With search, the prompt, and the LLM ready, we wire them together:

In [ ]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```

This approach is modular. You can swap the search backend, the prompt template, or the LLM model.

Nothing else needs to change. Later when we replace minsearch with sqlitesearch, only the `search` function changes.

In [ ]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

In [ ]:
answer = rag("How do I get a certificate?")
print(answer)

# Persistent Data Ingestion

- So far, our RAG pipeline loads data and builds the search index at startup.
- Minsearch is in-memory. It's a bunch of Python dictionaries bound to the process where it's running. When you stop the process, the data disappears, so you re-index every time you restart
    - Since our FAQ dataset is small, this is fine. The indexing takes less than a second and the entire pipeline runs in one process.
    - But, this breaks down as the dataset grows. Fetching data takes time - calling APIs, parsing files, cleaning text. With millions of documents, the startup becomes slow.
        - You don't want to wait minutes every time your service restarts.

- So we **separate ingestion from querying**. One process writes the data to a persistent search index. Another process reads from it. The two run independently and only share the index between them.
- The index survives restarts, so we ingest once and query as often as we like. This is the ingestion step from data engineering. We move data from its source into a target system the application can use.

- You can use any persistent search backend for this, such as Elasticsearch, OpenSearch, or Qdrant.
- We use [sqlitesearch](https://github.com/alexeygrigorev/sqlitesearch), a lightweight search library backed by SQLite FTS5. It has the same API as minsearch, so it's a drop-in replacement that happens to be persistent.

## Load docs

In [ ]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

## Index docs

In [ ]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="KB/faq.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

## Query docs

In [ ]:
print(index.count())

In [ ]:
results = index.search("Can I still join the course after it started?", num_results=5)
pprint([doc["question"] for doc in results])

## Choosing an approach

Pick the right tool for your data:

-   `minsearch`: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
-   `sqlitesearch`: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.

Use minsearch when you can load and index the data on startup without noticeable delay. Switch to a persistent backend when ingestion takes too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different backend:

-   Elasticsearch
-   OpenSearch
-   Qdrant (vector database)
-   Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

# Agents

Every pipeline we wrote so far followed the same flow:

-   search the FAQ,
-   build a prompt with the results,
-   send it to the LLM.

This returns good answers when the user's query matches the documents. The search finds the right entry, the LLM reads it, and you get a helpful reply.

Often, though, the search returns nothing useful.

-   Maybe the user made a typo.
-   Maybe they asked the question in an unusual way.
-   Maybe they need information from two different searches.

We use **lexical search** here, so the search looks for an exact match. One typo and it misses the entry it needed. In our pipeline there's no recovery. The search runs once, and if it returns garbage the LLM gets garbage. Our pipeline always does the same thing, no matter what.

Instead of routing the user question straight to search, we can hand control to the LLM and let it drive.

The LLM is in charge now, and it can:

-   fix typos
-   search again with different terms
-   ask the user a clarifying question

A fixed flow can't do any of this. Once we put the LLM in control, our system becomes agentic, so it's flexible rather than rigid.

An agent uses an LLM to decide which actions to take and in which order. Instead of a fixed flow, the LLM chooses what to do at each step.

We'll add the following to our pipeline:
-   Function calling, so we can give the LLM tools it can use
-   The agentic loop, where the LLM decides when to call a tool, when to call another one, and when to stop and answer
-   Frameworks, the libraries that run this loop for you
We build on top of the RAG pipeline we have, which uses keyword search with minsearch.

## A failing example without agents

In [ ]:
with open("templates/instructions.txt", "r") as file:
    instructions = file.read().strip()
with open("templates/prompt_template.txt", "r") as file:
    prompt_template = file.read().strip()

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="KB/llmzc_faq.db"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template
)

In [ ]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

In [ ]:
answer = assistant.rag("How do I run Olaamma locally?")
print(answer)

The flow that broke:

```mermaid
flowchart TD
    U([User: How do I run Olaamma?])
    S[search - Olaamma - no useful results]
    A([LLM: I don't have information about Olaamma.])

    U --> S --> A
```
- The pipeline is fixed: search, build prompt, LLM.
- The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

## Function Calling

- An agent puts the LLM in charge.
- Instead of running search ourselves, we give the LLM a `search` tool. It decides when to call it and what to search for.

The same typo question now goes like this:
```mermaid
flowchart TD
    U([User: How do I run Olaamma?])
    L1[LLM: I'll search for 'Olaamma']
    S1[search - Olaamma - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```
The LLM searched, saw the results were bad, and decided to try again with a different query. It made that decision on its own. We didn't write any code to handle typos.

The difference is about who makes the decisions:

-   With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
-   With an agent, the LLM decides. It chooses which actions to take and when to stop.

The mechanism that makes this possible is **function calling**.

### Asking without tools

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

### Defining the tool

In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

### Introduce the tool to the model

Next we tell the model about this function.

The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

The `description` is the most important field, because the model reads it to decide when to call the function. `parameters` is a JSON schema for the arguments, and we mark `query` as required so the model always fills it in.

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Sending the question with the tool

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)
response.output

- Look at the output. Instead of a message with the answer, the response contains a `function_call` entry. The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first.
- Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like ***"enrollment late join access"***.

### Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our `search` function, and serialize the result.

In [ ]:
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)
# pprint(result_json)

- Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.
- The `call_id` links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own `call_id`.
- We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as `input`. If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. That means the question, the decision to call `search`, and the result we got back.

In [ ]:
messages.extend(response.output)
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

### Asking the model again

We call the API a second time with the expanded history:

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

- That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.
- People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. The LLM decides which tools to call.

### Token usage and cost

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.75
    OUTPUT_PRICE_PER_MILLION = 4.5

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

In [ ]:
usage = response.usage
result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

## The Agentic Loop

- In the previous section, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.
- That works for one function call.
- It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. An agent is exactly that.

### Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

-   Instructions, the role and behavior we want. We pass this as the `developer` message. The better the instructions, the better the agent helps.
-   Tools, the functions the agent can call to carry out the task. For us that's only `search`.
-   Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

### A developer prompt

In [ ]:
with open("templates/agent_instructions.txt", "r") as f:
    instructions = f.read().strip()

print(instructions)

### A function-call helper

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    # When we add more tools later, we'll extend this with more if branches (or switch to a registry).

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

### Full agent loop function

- The `has_function_calls` flag tells us whether the model needs another API call.
- If the response contains a function call, the updated `messages` has tool output the model hasn't seen yet. We'll need to send it back.
- We wrap this in a `while` loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.
- The exit condition is the simplest one possible. No function calls this turn means we're done. Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
typo_question = "How do I run Olaamma locally?"
agent_loop(instructions, typo_question)

### Rejecting off-topic questions

In [ ]:
agent_loop(instructions, "what's queen gambit?")

## ToyAIKit

- The handwritten agent loop from the previous section is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.
- ToyAIKit wraps this pattern so we can focus on tools, prompts, and behavior. It does the same thing as our handwritten loop with less boilerplate. If you open its `runners` code, you'll find the same `while True` loop we wrote by hand.
- ToyAIKit is small and easy to read, so when something breaks you can see exactly what happened. That makes it handy for developing and debugging locally before we go to production.
- **One caveat**. ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. We use it because it's minimal and you can see what it does.

### Setup

In [ ]:
# !uv add toyaikit

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

### Registering the tool

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

### Letting ToyAIKit generate the schema

- Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.
- If we add a type hint and a docstring to `search`, ToyAIKit reads them and derives the schema for us.
- Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()

### The chat interface and runner

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

- The `chat_interface` handles display in the notebook.
- The `callback` renders model messages and tool calls as they happen.
- The runner runs the agent loop, the same `while True` we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.
- We pick `gpt-5.4-mini` here on purpose. Without it, ToyAIKit falls back to a smaller, faster default that doesn't follow the instructions as reliably.

### Running one prompt

In [ ]:
result = runner.loop(
    prompt=typo_question,
    callback=callback,
)

The `result` is a `LoopResult` with `all_messages` (the full conversation), token counts, and `cost` (computed from token usage).

### Cost and tokens

Look at what the call cost:

In [ ]:
result.cost

You can also look at the full message history.

In [ ]:
result.all_messages

In [ ]:
# runner.pricing_config.calculate_cost(
#     model="gpt-5.4-mini",
#     input_tokens=usage.input_tokens,
#     output_tokens=usage.output_tokens
# )
# calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)

### Continuing the conversation

- Take the messages from the previous result and pass them as `previous_messages` on the next `loop` call
- The runner picks up where the last call left off, with the same agent loop and an extended history. The model knows "different model" refers to Ollama because it sees the previous turn in memory. Without that history, it would have no idea what we're asking about.

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

### Interactive chat

- For a chat-like workflow, run the built-in input loop
- Type questions and get answers. Type "stop" to exit.

In [ ]:
runner.run()